In [1]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit_aer import Aer
from qiskit import transpile

In [2]:
def superdense_coding_register():
    Alice = QuantumRegister(1, "A")
    Bob = QuantumRegister(1, "B")
    cr = ClassicalRegister(2, "c")
    qc = QuantumCircuit(Alice, Bob, cr)
    
    return qc, cr, Alice, Bob

In [3]:
def create_entanglement(qc, Alice, Bob):
    qc.h(Alice)
    qc.cx(Alice, Bob)

    qc.barrier()

    return qc

In [4]:
def encode_information_alice(qc, Alice, message):
    if len(message) != 2 or any(bit not in ["0", "1"] for bit in message):
        raise ValueError()

    if message[1] == "1":
        qc.x(Alice)
    if message[0] == "1":
        qc.z(Alice)

    qc.barrier()

    return qc

In [5]:
def decode_bob(qc, cr, Alice, Bob):
    qc.cx(Alice, Bob)
    qc.h(Alice)

    qc.measure(Alice, cr[0])
    qc.measure(Bob, cr[1])

    return qc

In [6]:
def run_superdense_simulation(message="11"):
    qc, cr, Alice, Bob = superdense_coding_register()
    create_entanglement(qc, Alice, Bob)
    
    try:
        encode_information_alice(qc, Alice, message)
    except ValueError as e:
            print(f"error: {e}")
            return None
    
    decode_bob(qc, cr, Alice, Bob)

    return qc

In [7]:
backend = Aer.get_backend("aer_simulator")
circuit = run_superdense_simulation("11")
compiled = transpile(circuit, backend)
job = backend.run(compiled, shots=1024)
result = job.result()
counts = result.get_counts()
received = list(counts.keys())[0]
print(f"Sent: 11, Received: {received}")

Sent: 11, Received: 11


In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

def make_noise_model(p_1q, p_2q, p_readout):
    noise = NoiseModel()

    err_1q = depolarizing_error(p_1q, 1)
    err_2q = depolarizing_error(p_2q, 2)

    for gate in ["h", "x", "z"]:
        noise.add_all_qubit_quantum_error(err_1q, gate)
    noise.add_all_qubit_quantum_error(err_2q, "cx")

    readout_error = ReadoutError([[1 - p_readout, p_readout], [p_readout, 1 - p_readout]])
    noise.add_all_qubit_readout_error(readout_error)

    return noise


In [ ]:
# Run ideal vs noisy simulation

message = "11"
shots = 2048

ideal_backend = AerSimulator()
noisy_backend = AerSimulator(noise_model=make_noise_model(p_1q=0.01, p_2q=0.04, p_readout=0.02))

circuit = run_superdense_simulation(message)

ideal_job = ideal_backend.run(transpile(circuit, ideal_backend), shots=shots)
noisy_job = noisy_backend.run(transpile(circuit, noisy_backend), shots=shots)

ideal_counts = ideal_job.result().get_counts()
noisy_counts = noisy_job.result().get_counts()

print(f"Sent: {message}")
print("Ideal counts:", ideal_counts)
print("Noisy counts:", noisy_counts)
print(f"Noisy accuracy for {message}: {noisy_counts.get(message, 0) / shots:.3f}")


Sent: 11
Ideal counts: {'11': 2048}
Noisy counts: {'01': 84, '00': 39, '10': 93, '11': 1832}
Noisy accuracy for 11: 0.895


In [ ]:
# Run on hardware

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService(channel="ibm_quantum_platform", instance="eburdak")
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
print(f"Chosen backend: {backend.name}")

message = "11"
circuit = run_superdense_simulation(message)
compiled = transpile(circuit, backend)

sampler = Sampler(mode=backend)
job = sampler.run([compiled], shots=1024)
pub_result = job.result()[0]

counts = pub_result.data.cr.get_counts()
print("Hardware counts:", counts)


qiskit_runtime_service.__init__:WARNING:2026-03-02 08:00:27,395: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: eburdak. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-02 08:00:28,724: Loading instance: eburdak, plan: open
qiskit_runtime_service.backends:WARNING:2026-03-02 08:00:32,251: Using instance: eburdak, plan: open


Chosen backend: ibm_fez
Hardware counts: {'11': 981, '01': 35, '10': 7, '00': 1}
